# Transformer (STFTnet-inspired) Gesture Classifier

## Overview

This notebook implements a **Transformer-based dynamic gesture recognition** model for data-glove time-series data, inspired by the architecture described in:

> *"A Convolutional-Transformer-Based Approach for Dynamic Gesture Recognition of Data Gloves"* (STFTnet), which achieved **99.72% accuracy** on their benchmark dataset.

---

## Why Transformers for Gesture Sequences?

Each glove recording is a **multivariate time series** — hundreds of IMU and flex sensor channels sampled over the duration of a gesture. The fundamental modelling challenge is relating sensor measurements across time to recognise the gesture.

**RNNs / LSTMs** read sequences left-to-right and maintain a hidden state, creating a *sequential bottleneck*: information from early timesteps must pass through many hidden-state updates to influence the final prediction, which can lead to vanishing gradients and difficulty capturing long-range dependencies.

**Transformers** solve this with **self-attention**: every timestep attends *directly* to every other timestep in a single operation, regardless of their distance in the sequence. The model learns to weight which (time, feature) combinations are most informative for recognising each gesture.

For a gesture like a 'wave', the model can directly relate the hand's orientation at the beginning of the wave to its position at the end — without relying on gated memory across many steps.

---

## Architecture Pipeline

```
Input (batch, T, C)
      │
      ▼
[MFA Block] ─── Multi-sensor Feature Attention
  GAP over time → Dense(γ, relu) → Dense(C, sigmoid) → element-wise gate
      │
      ▼
[Depthwise Separable Conv1D] ─── Feature Embedding
  Depthwise Conv → Pointwise Conv → projects C → D_MODEL
      │
      ▼
[Positional Encoding] ─── Sinusoidal injection of position information
      │
      ▼
[Transformer Encoder × N_LAYERS]
  MultiHeadAttention → Add & Norm
  FFN (Dense → ReLU → Dense) → Add & Norm
      │
      ▼
[GlobalAveragePooling1D] ─── Aggregate temporal context
      │
      ▼
[Dense Head + Dropout] → Softmax
```

The **MFA (Multi-sensor Feature Attention) block** is a channel-attention mechanism: it learns which sensor channels are most discriminative for gesture recognition, suppressing noisy sensors and amplifying informative ones.

The **depth-wise separable convolution** efficiently embeds the raw (filtered, resampled) features into the Transformer's `D_MODEL`-dimensional token space with far fewer parameters than a standard dense projection.

---

## Key Hyperparameters (from paper)

| Parameter | Paper | This notebook |
|---|---|---|
| `D_MODEL` | 64 | 64 |
| `N_LAYERS` | 3 | 3 |
| `N_HEADS` | 16 | 4 (auto-adjusts to feature count) |
| `MFA_GAMMA` | 256 | 64 (scaled to sensor count) |
| `LR` | 0.001 | 0.001 |
| `batch_size` | 64 | 32 |
| `weight_decay` | 1e-8 | 1e-8 (AdamW) |
| Sequence length | 50 steps | 110 steps |

In [ ]:
# Cell 1 — Install
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

pip_install("tensorflow")
pip_install("scikit-learn")
pip_install("matplotlib")
pip_install("seaborn")
pip_install("pandas")
pip_install("numpy")
pip_install("scipy")

# Optional: tensorflow-addons for AdamW (may not be available for all TF versions)
try:
    pip_install("tensorflow-addons")
    print("tensorflow-addons installed.")
except Exception:
    print("tensorflow-addons unavailable — will use Adam with weight_decay note.")

print("Installation complete.")

In [ ]:
# Cell 2 — Imports
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")

# ── TensorFlow ──────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
print(f"TensorFlow version: {tf.__version__}")

# ── tensorflow-addons (AdamW) ───────────────────────────────────────────────
TFA_AVAILABLE = False
try:
    import tensorflow_addons as tfa
    TFA_AVAILABLE = True
    print("tensorflow-addons available — will use AdamW.")
except ImportError:
    print("tensorflow-addons not available — will use legacy Adam (weight_decay not applied).")

# ── sklearn ─────────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix

# ── Project utilities ────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(os.path.abspath(''))
sys.path.insert(0, str(NOTEBOOK_DIR))
from utils.data_loader import (
    build_dataset, split_dataset, build_column_groups,
    FINGERS, HANDS, IMU_CHANNELS, FLEX_CHANNELS,
    PALM_CHANNELS, WRIST_CHANNELS
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F7F6F2',
    'axes.edgecolor': '#D4D1CA',
    'axes.labelcolor': '#28251D',
    'xtick.color': '#28251D',
    'ytick.color': '#28251D',
    'text.color': '#28251D',
    'font.family': 'sans-serif',
    'grid.color': '#D4D1CA',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
})
TEAL     = '#20808D'
RUST     = '#A84B2F'
DARK_TEAL = '#1B474D'

print("Imports complete.")

In [ ]:
# Cell 3 — CONFIG

# ── DATA ──────────────────────────────────────────────────────────────────────
DATA_ROOT = '../data'
FS_HZ     = 22.0

# ── COLUMN SELECTION ──────────────────────────────────────────────────────────
USE_LEFT_HAND     = True
USE_RIGHT_HAND    = True
USE_DISTAL_IMU    = True
USE_PROXIMAL_IMU  = True
USE_ACCELEROMETER = True
USE_ORIENTATION   = True
USE_FLEX_SENSORS  = True
USE_PALM_IMU      = True
USE_WRIST_IMU     = True

# ── PREPROCESSING ─────────────────────────────────────────────────────────────
FILTER_TYPE     = 'butterworth_lp'
TARGET_LEN      = 110           # Resampled sequence length
NORMALIZATION   = 'minmax'
PER_SAMPLE_NORM = True

# ── DATASET SPLIT ─────────────────────────────────────────────────────────────
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.10
RANDOM_SEED = 42

# ── MODEL HYPERPARAMETERS ─────────────────────────────────────────────────────
# Multi-sensor Feature Attention (MFA) block
USE_MFA_BLOCK = True
MFA_GAMMA     = 64     # Increased dimension in MFA Conv1d (paper uses 256, reduce for fewer features)

# Feature embedding (depth-wise separable conv)
D_MODEL       = 64     # Transformer model dimension

# Transformer encoder
N_LAYERS      = 3      # Number of transformer encoder blocks
N_HEADS       = 4      # Number of attention heads (D_MODEL must be divisible by N_HEADS)
FFN_DIM       = 128    # Feed-forward network hidden dimension (usually 2-4× D_MODEL)
TRANSFORMER_DROPOUT = 0.1

# Positional encoding
USE_POSITIONAL_ENCODING = True

# Classification head
DENSE_UNITS   = [64]
DROPOUT_RATE  = 0.3

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS              = 80
BATCH_SIZE          = 32
LEARNING_RATE       = 1e-3
WEIGHT_DECAY        = 1e-8    # Applied via AdamW
EARLY_STOP_PATIENCE = 15
LR_WARMUP_STEPS     = 100     # Set 0 to disable warmup

print("CONFIG set.")
print(f"  Data root  : {DATA_ROOT}")
print(f"  Target len : {TARGET_LEN} timesteps")
print(f"  D_MODEL    : {D_MODEL}, N_HEADS: {N_HEADS}, N_LAYERS: {N_LAYERS}")

In [ ]:
# Cell 4 — Build feature columns from CONFIG

# ── Determine active hands ────────────────────────────────────────────────────
active_hands = []
if USE_LEFT_HAND:  active_hands.append('left')
if USE_RIGHT_HAND: active_hands.append('right')

# ── Determine active IMU locations ────────────────────────────────────────────
active_locs = []
if USE_DISTAL_IMU:   active_locs.append('mid')
if USE_PROXIMAL_IMU: active_locs.append('prox')

# ── Determine active IMU channels ─────────────────────────────────────────────
active_imu_channels = []
if USE_ACCELEROMETER: active_imu_channels += ['ax', 'ay', 'az']
if USE_ORIENTATION:   active_imu_channels += ['pitch', 'roll', 'yaw']

# ── Build feature columns ─────────────────────────────────────────────────────
feature_cols = []

for hand in active_hands:
    for finger in FINGERS:
        for loc in active_locs:
            for ch in active_imu_channels:
                feature_cols.append(f"{hand}_{finger}_{loc}_{ch}")
        if USE_FLEX_SENSORS:
            feature_cols.append(f"{hand}_{finger}_flex_mcp")
            feature_cols.append(f"{hand}_{finger}_flex_pip")
    if USE_PALM_IMU:
        for ch in active_imu_channels:
            feature_cols.append(f"{hand}_palm_{ch}")
    if USE_WRIST_IMU:
        for ch in active_imu_channels:
            feature_cols.append(f"{hand}_wrist_{ch}")

print(f"Expected feature columns : {len(feature_cols)}")
print("\nSample columns (first 15):")
for c in feature_cols[:15]:
    print(f"  {c}")
if len(feature_cols) > 15:
    print(f"  ... (+{len(feature_cols) - 15} more)")

In [ ]:
# Cell 5 — Load dataset, class distribution

print("Loading dataset — this may take a minute...")
X, y, labels, feature_cols_used = build_dataset(
    data_root       = DATA_ROOT,
    feature_cols    = feature_cols,
    filter_type     = FILTER_TYPE,
    fs              = FS_HZ,
    target_len      = TARGET_LEN,
    normalization   = NORMALIZATION,
    per_sample_norm = PER_SAMPLE_NORM,
    exclude_quat    = True,
)

N_CLASSES   = len(labels)
N_FEATURES  = X.shape[2]
N_TIMESTEPS = X.shape[1]

print(f"\nDataset shape : {X.shape}  (samples × timesteps × features)")
print(f"Classes       : {N_CLASSES}")
print(f"Features used : {N_FEATURES} (requested {len(feature_cols)})")
print(f"y range       : [{y.min()}, {y.max()}]")

# ── Class distribution plot ───────────────────────────────────────────────────
counts = pd.Series(y).value_counts().sort_index()
label_names = [labels[i] for i in counts.index]

fig, ax = plt.subplots(figsize=(max(8, N_CLASSES * 0.6), 4))
bars = ax.bar(range(len(counts)), counts.values, color=TEAL, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(label_names, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Gesture Class')
ax.set_ylabel('Sample Count')
ax.set_title('Class Distribution', fontsize=13, fontweight='bold', pad=10)

# Annotate bar counts
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            str(count), ha='center', va='bottom', fontsize=8, color='#28251D')

plt.tight_layout()
plt.show()

print("\nLabel mapping:")
for i, lbl in enumerate(labels):
    print(f"  {i:>3}  {lbl}")

In [ ]:
# Cell 6 — Split dataset

(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_dataset(
    X, y,
    train_ratio = TRAIN_RATIO,
    val_ratio   = VAL_RATIO,
    seed        = RANDOM_SEED,
)

print(f"\nTrain : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}")

# ── Verify class balance in each split ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=False)
splits    = [(y_train, 'Train', TEAL), (y_val, 'Val', RUST), (y_test, 'Test', DARK_TEAL)]

for ax, (y_split, split_name, color) in zip(axes, splits):
    cnts = pd.Series(y_split).value_counts().sort_index()
    ax.bar(range(len(cnts)), cnts.values, color=color, edgecolor='white')
    ax.set_title(f'{split_name} ({len(y_split)} samples)', fontweight='bold')
    ax.set_xlabel('Class index')
    ax.set_ylabel('Count')
    ax.set_xticks(range(len(cnts)))
    ax.set_xticklabels(range(len(cnts)), fontsize=7)

plt.suptitle('Class Distribution per Split', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Cell 7 — Model Definition

The model is built using the **TensorFlow/Keras functional API** with four custom layer classes:

### 1. MFABlock (Multi-sensor Feature Attention)
Inspired by Squeeze-and-Excitation networks adapted for temporal sensor data. It globally averages over time to produce a channel descriptor, passes it through two dense layers (a bottleneck then a gating layer), and applies the resulting per-channel weights as a multiplicative gate on the input. This suppresses uninformative sensors.

### 2. DepthwiseSeparableConv1D
Factorises a standard Conv1D into a **depthwise** convolution (applied per channel, capturing local temporal patterns) followed by a **pointwise** Conv1D (1×1, projects to `D_MODEL`). This reduces parameter count while retaining representational power.

### 3. PositionalEncoding
Since self-attention is permutation-invariant, position must be injected explicitly. We use sinusoidal encodings:
\[ PE(pos, 2i) = \sin(pos / 10000^{2i/d}) \]
\[ PE(pos, 2i+1) = \cos(pos / 10000^{2i/d}) \]
These are added to the embedded features before the Transformer encoder stack.

### 4. TransformerEncoderBlock
Each block implements the standard **Pre-Norm** Transformer encoder:
- Multi-Head Self-Attention with residual connection and Layer Normalisation
- Position-wise Feed-Forward Network (two dense layers with ReLU) with residual connection and Layer Normalisation

Per the STFTnet paper, we use **BatchNorm** in an alternative variant but default to **LayerNorm** for stability.

In [ ]:
# Cell 7 — Model definition (TensorFlow/Keras functional API, custom layers)

# ─────────────────────────────────────────────────────────────────────────────
# Auto-adjust N_HEADS so D_MODEL is divisible
# ─────────────────────────────────────────────────────────────────────────────
def find_valid_n_heads(d_model, requested_heads):
    """Return the nearest valid number of attention heads <= requested_heads."""
    if d_model % requested_heads == 0:
        return requested_heads
    # Try smaller values first, then larger
    for h in range(requested_heads, 0, -1):
        if d_model % h == 0:
            print(f"[WARNING] N_HEADS={requested_heads} does not divide D_MODEL={d_model}. "
                  f"Auto-adjusted N_HEADS → {h}")
            return h
    return 1

N_HEADS_ACTUAL = find_valid_n_heads(D_MODEL, N_HEADS)


# ─────────────────────────────────────────────────────────────────────────────
# 1. MFABlock — Multi-sensor Feature Attention
# ─────────────────────────────────────────────────────────────────────────────
class MFABlock(keras.layers.Layer):
    """
    Multi-sensor Feature Attention Block.

    Implements a channel-attention mechanism:
      1. Global Average Pooling across time → (batch, C)
      2. Dense bottleneck (gamma units, ReLU)
      3. Dense gate (C units, Sigmoid)  → per-channel weights in [0, 1]
      4. Element-wise multiply: gate × input  → attended (batch, T, C)

    This lets the network learn which sensors are most discriminative
    for each gesture class.
    """

    def __init__(self, gamma, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma

    def build(self, input_shape):
        n_features = input_shape[-1]
        self.gap  = keras.layers.GlobalAveragePooling1D()
        self.fc1  = keras.layers.Dense(self.gamma, activation='relu',
                                       name='mfa_fc1')
        self.fc2  = keras.layers.Dense(n_features, activation='sigmoid',
                                       name='mfa_fc2')
        self.mul  = keras.layers.Multiply()
        super().build(input_shape)

    def call(self, x):
        # x: (batch, T, C)
        gap  = self.gap(x)        # (batch, C)
        gate = self.fc1(gap)      # (batch, gamma)
        gate = self.fc2(gate)     # (batch, C)
        # Expand gate to broadcast over time dimension
        gate = tf.expand_dims(gate, axis=1)  # (batch, 1, C)
        return x * gate           # (batch, T, C)

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 2. DepthwiseSeparableConv1D — Local temporal embedding → D_MODEL
# ─────────────────────────────────────────────────────────────────────────────
class DepthwiseSeparableConv1D(keras.layers.Layer):
    """
    Depthwise Separable Conv1D embedding layer.

    Depthwise conv: applies one filter per input channel (local temporal patterns)
    Pointwise conv: 1×1 convolution to project to d_model dimensions

    Significantly fewer parameters than a standard Conv1D projection.
    """

    def __init__(self, d_model, kernel_size=3, **kwargs):
        super().__init__(**kwargs)
        self.d_model     = d_model
        self.kernel_size = kernel_size

    def build(self, input_shape):
        # Depth-wise: separate filter per channel, kernel_size temporal window
        self.dw_conv = keras.layers.DepthwiseConv2D(
            kernel_size=(1, self.kernel_size),
            padding='same',
            activation='relu',
            name='dw_conv'
        )
        # Pointwise: 1×1 projection to d_model
        self.pw_conv = keras.layers.Conv1D(
            filters=self.d_model,
            kernel_size=1,
            padding='same',
            activation='relu',
            name='pw_conv'
        )
        self.bn = keras.layers.BatchNormalization(name='dw_sep_bn')
        super().build(input_shape)

    def call(self, x, training=False):
        # x: (batch, T, C)
        # DepthwiseConv2D expects (batch, H, W, C), so expand dims
        x_4d = tf.expand_dims(x, axis=1)         # (batch, 1, T, C)
        x_dw = self.dw_conv(x_4d)                # (batch, 1, T, C)  — depthwise
        x_3d = tf.squeeze(x_dw, axis=1)          # (batch, T, C)
        x_pw = self.pw_conv(x_3d)                # (batch, T, d_model)
        return self.bn(x_pw, training=training)

    def get_config(self):
        config = super().get_config()
        config.update({'d_model': self.d_model, 'kernel_size': self.kernel_size})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 3. PositionalEncoding — Sinusoidal injection of position information
# ─────────────────────────────────────────────────────────────────────────────
class PositionalEncoding(keras.layers.Layer):
    """
    Sinusoidal positional encoding.

    Since self-attention is permutation-invariant, positional information must
    be injected. We add fixed sinusoidal encodings of each timestep position
    to the embedded features, following Vaswani et al. (2017).

    PE(pos, 2i)   = sin(pos / 10000^(2i/d))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
    """

    def __init__(self, max_len=512, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len

    def build(self, input_shape):
        d = input_shape[-1]       # d_model dimension
        T = min(input_shape[-2] or self.max_len, self.max_len)

        positions = np.arange(T)[:, np.newaxis]          # (T, 1)
        dims      = np.arange(d)[np.newaxis, :]           # (1, d)
        angles    = positions / np.power(
            10000.0, (2 * (dims // 2)) / np.float32(d)
        )  # (T, d)

        # sin to even indices, cos to odd
        angles[:, 0::2] = np.sin(angles[:, 0::2])
        angles[:, 1::2] = np.cos(angles[:, 1::2])

        # (1, T, d) — broadcast over batch
        self.pe = tf.constant(angles[np.newaxis, :, :], dtype=tf.float32)
        super().build(input_shape)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return x + self.pe[:, :seq_len, :]

    def get_config(self):
        config = super().get_config()
        config.update({'max_len': self.max_len})
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 4. TransformerEncoderBlock
# ─────────────────────────────────────────────────────────────────────────────
class TransformerEncoderBlock(keras.layers.Layer):
    """
    Standard Transformer encoder block:

      x' = LayerNorm(x + MultiHeadSelfAttention(x))
      x'' = LayerNorm(x' + FFN(x'))

    FFN: Dense(ffn_dim, relu) → Dense(d_model)

    Also exposes `last_attn_weights` after each forward pass
    for attention visualisation.
    """

    def __init__(self, d_model, n_heads, ffn_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.d_model      = d_model
        self.n_heads      = n_heads
        self.ffn_dim      = ffn_dim
        self.dropout_rate = dropout_rate

        self.mha    = keras.layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model // n_heads,
            dropout=dropout_rate, name='mha'
        )
        self.norm1  = keras.layers.LayerNormalization(epsilon=1e-6, name='ln1')
        self.norm2  = keras.layers.LayerNormalization(epsilon=1e-6, name='ln2')
        self.ffn1   = keras.layers.Dense(ffn_dim, activation='relu', name='ffn1')
        self.ffn2   = keras.layers.Dense(d_model, name='ffn2')
        self.drop1  = keras.layers.Dropout(dropout_rate)
        self.drop2  = keras.layers.Dropout(dropout_rate)

        # Stores attention weights after last forward pass (for visualisation)
        self.last_attn_weights = None

    def call(self, x, training=False):
        # Multi-Head Self-Attention sub-layer
        attn_out, attn_weights = self.mha(
            x, x, return_attention_scores=True, training=training
        )
        self.last_attn_weights = attn_weights   # (batch, heads, T, T)
        attn_out = self.drop1(attn_out, training=training)
        x = self.norm1(x + attn_out)

        # Feed-Forward sub-layer
        ffn_out = self.ffn1(x)
        ffn_out = self.ffn2(ffn_out)
        ffn_out = self.drop2(ffn_out, training=training)
        x = self.norm2(x + ffn_out)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'd_model': self.d_model, 'n_heads': self.n_heads,
            'ffn_dim': self.ffn_dim, 'dropout_rate': self.dropout_rate
        })
        return config


# ─────────────────────────────────────────────────────────────────────────────
# 5. Build the full model (functional API)
# ─────────────────────────────────────────────────────────────────────────────
def build_transformer_model(
        n_timesteps, n_features, n_classes,
        d_model=64, n_layers=3, n_heads=4,
        ffn_dim=128, mfa_gamma=64,
        dense_units=None, dropout_rate=0.3,
        transformer_dropout=0.1,
        use_mfa=True, use_pos_enc=True,
):
    """
    Build the STFTnet-inspired Transformer gesture classifier.

    Architecture:
      Input → [MFABlock] → DepthwiseSepConv → [PosEnc]
            → N × TransformerEncoder → GAP → Dense head → Softmax
    """
    if dense_units is None:
        dense_units = [64]

    inputs = keras.Input(shape=(n_timesteps, n_features), name='input')
    x = inputs

    # ── MFA Block ────────────────────────────────────────────────────────────
    if use_mfa:
        x = MFABlock(gamma=mfa_gamma, name='mfa_block')(x)

    # ── Depth-wise Separable Conv embedding ──────────────────────────────────
    x = DepthwiseSeparableConv1D(d_model=d_model, kernel_size=3,
                                  name='dw_sep_embed')(x)

    # ── Positional Encoding ───────────────────────────────────────────────────
    if use_pos_enc:
        x = PositionalEncoding(max_len=n_timesteps + 64, name='pos_enc')(x)

    # ── Transformer Encoder Stack ─────────────────────────────────────────────
    for i in range(n_layers):
        x = TransformerEncoderBlock(
            d_model=d_model, n_heads=n_heads, ffn_dim=ffn_dim,
            dropout_rate=transformer_dropout,
            name=f'transformer_block_{i}'
        )(x)

    # ── Aggregate temporal context ────────────────────────────────────────────
    x = keras.layers.GlobalAveragePooling1D(name='gap')(x)

    # ── Classification Head ───────────────────────────────────────────────────
    for i, units in enumerate(dense_units):
        x = keras.layers.Dense(units, activation='relu',
                                name=f'head_dense_{i}')(x)
        x = keras.layers.Dropout(dropout_rate, name=f'head_drop_{i}')(x)

    outputs = keras.layers.Dense(n_classes, activation='softmax',
                                  name='output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='STFTnet_Transformer')
    return model


# ── Instantiate ───────────────────────────────────────────────────────────────
model = build_transformer_model(
    n_timesteps       = N_TIMESTEPS,
    n_features        = N_FEATURES,
    n_classes         = N_CLASSES,
    d_model           = D_MODEL,
    n_layers          = N_LAYERS,
    n_heads           = N_HEADS_ACTUAL,
    ffn_dim           = FFN_DIM,
    mfa_gamma         = MFA_GAMMA,
    dense_units       = DENSE_UNITS,
    dropout_rate      = DROPOUT_RATE,
    transformer_dropout = TRANSFORMER_DROPOUT,
    use_mfa           = USE_MFA_BLOCK,
    use_pos_enc       = USE_POSITIONAL_ENCODING,
)

# ── Optimiser ────────────────────────────────────────────────────────────────
if TFA_AVAILABLE:
    optimizer = tfa.optimizers.AdamW(
        learning_rate = LEARNING_RATE,
        weight_decay  = WEIGHT_DECAY,
    )
    print(f"Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
else:
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    print(f"Optimizer: Adam (lr={LEARNING_RATE})  — weight_decay={WEIGHT_DECAY} not applied")

model.compile(
    optimizer = optimizer,
    loss      = 'sparse_categorical_crossentropy',
    metrics   = ['accuracy'],
)

model.summary()

In [ ]:
# Cell 8 — Training with callbacks + learning curve plots

# ── Paths ─────────────────────────────────────────────────────────────────────
RESULTS_DIR    = Path('results')
SAVED_MODELS_DIR = Path('saved_models')
RESULTS_DIR.mkdir(exist_ok=True)
SAVED_MODELS_DIR.mkdir(exist_ok=True)

CKPT_PATH = SAVED_MODELS_DIR / 'transformer_best.keras'

# ── LR Schedule (linear warmup + cosine decay) ────────────────────────────────
class WarmupCosineSchedule(keras.callbacks.Callback):
    """
    Linear warmup for LR_WARMUP_STEPS steps, then follows Adam's
    internal learning rate (i.e., returns to LEARNING_RATE smoothly).
    This avoids large gradient updates from random initialisation.
    """
    def __init__(self, warmup_steps, base_lr):
        super().__init__()
        self.warmup_steps = warmup_steps
        self.base_lr      = base_lr
        self.step         = 0

    def on_train_batch_begin(self, batch, logs=None):
        if self.warmup_steps <= 0:
            return
        self.step += 1
        if self.step <= self.warmup_steps:
            lr = self.base_lr * (self.step / self.warmup_steps)
            K.set_value(self.model.optimizer.lr, lr)


# ── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath        = str(CKPT_PATH),
        monitor         = 'val_accuracy',
        save_best_only  = True,
        save_format     = 'keras',
        verbose         = 1,
    ),
    keras.callbacks.EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = EARLY_STOP_PATIENCE,
        restore_best_weights = True,
        verbose              = 1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 7,
        min_lr   = 1e-6,
        verbose  = 1,
    ),
    keras.callbacks.CSVLogger(
        str(RESULTS_DIR / 'transformer_training_log.csv')
    ),
]

if LR_WARMUP_STEPS > 0:
    callbacks.append(WarmupCosineSchedule(LR_WARMUP_STEPS, LEARNING_RATE))
    print(f"LR warmup enabled: {LR_WARMUP_STEPS} steps")

# ── Train ─────────────────────────────────────────────────────────────────────
print(f"\nTraining for up to {EPOCHS} epochs (batch={BATCH_SIZE}, early stop patience={EARLY_STOP_PATIENCE})...")
history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    callbacks       = callbacks,
    verbose         = 1,
)

# ── Learning Curves ───────────────────────────────────────────────────────────
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss
axes[0].plot(epochs_ran, hist['loss'],     color=TEAL,      lw=2, label='Train loss')
axes[0].plot(epochs_ran, hist['val_loss'], color=RUST,      lw=2, label='Val loss', linestyle='--')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Sparse Categorical Crossentropy')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(epochs_ran, hist['accuracy'],     color=TEAL, lw=2, label='Train acc')
axes[1].plot(epochs_ran, hist['val_accuracy'], color=RUST, lw=2, label='Val acc', linestyle='--')
axes[1].set_title('Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim([0, 1.05])
axes[1].legend()
axes[1].grid(True)

# Annotate best val accuracy
best_val_acc = max(hist['val_accuracy'])
best_ep      = hist['val_accuracy'].index(best_val_acc) + 1
axes[1].axvline(best_ep, color=DARK_TEAL, lw=1, linestyle=':')
axes[1].text(best_ep + 0.3, best_val_acc - 0.04,
             f'Best={best_val_acc:.4f}\n(ep {best_ep})',
             fontsize=8, color=DARK_TEAL)

plt.suptitle('Training Curves — STFTnet Transformer', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nBest val accuracy : {best_val_acc:.4f}  (epoch {best_ep})")

In [ ]:
# Cell 9 — Evaluation: classification report + confusion matrix

# ── Reload best checkpoint ────────────────────────────────────────────────────
if CKPT_PATH.exists():
    model = keras.models.load_model(
        str(CKPT_PATH),
        custom_objects={
            'MFABlock': MFABlock,
            'DepthwiseSeparableConv1D': DepthwiseSeparableConv1D,
            'PositionalEncoding': PositionalEncoding,
            'TransformerEncoderBlock': TransformerEncoderBlock,
        }
    )
    print("Loaded best checkpoint.")

# ── Test set evaluation ───────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")

# ── Predictions ───────────────────────────────────────────────────────────────
y_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

# ── Classification Report ─────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=labels, digits=4))

# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

# Normalise by true class (rows)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(12, N_CLASSES * 0.9), max(8, N_CLASSES * 0.7)))

# Raw counts
sns.heatmap(
    cm, ax=axes[0],
    annot=(N_CLASSES <= 20), fmt='d', cmap='YlOrRd',
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, cbar=True
)
axes[0].set_title('Confusion Matrix (counts)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=45)

# Normalised
sns.heatmap(
    cm_norm, ax=axes[1],
    annot=(N_CLASSES <= 20), fmt='.2f', cmap='YlOrBr',
    xticklabels=labels, yticklabels=labels,
    vmin=0, vmax=1, linewidths=0.5, cbar=True
)
axes[1].set_title('Confusion Matrix (normalised)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle(f'Test Accuracy: {test_acc:.4f}', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Per-class summary ─────────────────────────────────────────────────────────
per_class_acc = cm.diagonal() / cm.sum(axis=1)
worst_classes = np.argsort(per_class_acc)[:5]
print("\nLowest per-class accuracy:")
for idx in worst_classes:
    print(f"  {labels[idx]:>30s}  {per_class_acc[idx]:.4f}")

## Cell 10 — Attention Visualisation

One of the key interpretability advantages of Transformers is that attention weights are directly inspectable. Each attention head computes a \(T \times T\) matrix (where \(T\) is the number of timesteps), representing how much each timestep *attends to* every other timestep when building its contextual representation.

**Reading the heatmap:**
- Row \(i\), column \(j\): the weight that timestep \(i\) assigns to timestep \(j\) when computing its output.
- A bright diagonal indicates the model attends locally (each moment mostly attends to itself/neighbours).
- Off-diagonal bright patches indicate global temporal dependencies — e.g., the model relates the start of a gesture to its end.
- Different heads typically specialise in different patterns (local vs. global, different phases of the gesture).

We extract weights from **the last Transformer encoder block**, as it captures the highest-level temporal abstractions.

In [ ]:
# Cell 10 — Attention weight visualisation

# ── Build an attention-extraction sub-model ───────────────────────────────────
# We need a model that outputs BOTH predictions AND the attention weights
# from each TransformerEncoderBlock.

# Strategy: rebuild the forward pass collecting attention outputs via a
# custom model that exposes the internal layers.

def get_attention_extractor(model):
    """
    Create a tf.keras.Model that runs a single forward pass and returns
    attention weights from all TransformerEncoderBlock layers.

    We extract weights by temporarily wrapping the MHA layers to capture outputs.
    """
    transformer_blocks = [
        layer for layer in model.layers
        if isinstance(layer, TransformerEncoderBlock)
    ]
    if not transformer_blocks:
        raise RuntimeError("No TransformerEncoderBlock found in model.")

    # Build an attention sub-model: input → each MHA layer's attention output
    inp = model.input

    # We'll use the Keras functional graph to find attention outputs.
    # Each TransformerEncoderBlock.mha is a MultiHeadAttention layer that
    # already stores attention scores when called with return_attention_scores=True.
    # We re-run inference and intercept via a custom Model.
    return transformer_blocks


transformer_blocks = get_attention_extractor(model)
print(f"Found {len(transformer_blocks)} TransformerEncoderBlock(s) in model.")

# ── Pick a test sample per class for visualisation ────────────────────────────
# Use one sample from the test set for a meaningful visualisation
# Pick the first correctly-classified sample
correct_mask = (y_pred == y_test)
first_correct_idx = np.where(correct_mask)[0][0] if correct_mask.any() else 0

viz_sample = X_test[first_correct_idx:first_correct_idx+1]   # (1, T, C)
true_label  = labels[y_test[first_correct_idx]]
pred_label  = labels[y_pred[first_correct_idx]]

print(f"\nVisualisation sample:  true='{true_label}',  predicted='{pred_label}'")

# ── Run one forward pass and capture attention weights ────────────────────────
# We use a sub-model built on the fly from the model's layers
def extract_attention_weights(model, x, block_idx=None):
    """
    Run a forward pass through the model, capturing attention weights
    from all TransformerEncoderBlock layers via their .last_attn_weights attribute.

    Returns: list of (n_heads, T, T) arrays, one per block.
    """
    blocks = [
        layer for layer in model.layers
        if isinstance(layer, TransformerEncoderBlock)
    ]

    # Forward pass — blocks store weights in .last_attn_weights during call()
    _ = model(x, training=False)

    attn_list = []
    for blk in blocks:
        if blk.last_attn_weights is not None:
            w = blk.last_attn_weights.numpy()  # (1, n_heads, T, T)
            attn_list.append(w[0])             # (n_heads, T, T)
    return attn_list


attn_weights_list = extract_attention_weights(model, viz_sample)
n_blocks = len(attn_weights_list)
print(f"Extracted attention weights from {n_blocks} block(s).")
if n_blocks > 0:
    print(f"Shape per block: {attn_weights_list[0].shape}  (heads × T × T)")

# ── Visualise: per-head attention map from the last block ─────────────────────
last_attn = attn_weights_list[-1]    # (n_heads, T, T) from last encoder block
n_heads_actual = last_attn.shape[0]

# Mean over heads (overall attention pattern)
mean_attn = last_attn.mean(axis=0)   # (T, T)

# ── Figure 1: Mean attention + head-by-head grid ──────────────────────────────
ncols = min(n_heads_actual, 4)
nrows = 1 + int(np.ceil(n_heads_actual / ncols))

fig = plt.figure(figsize=(4 * ncols, 3.5 * nrows))
gs  = gridspec.GridSpec(nrows, ncols, figure=fig)

# Top row: mean attention
ax_mean = fig.add_subplot(gs[0, :ncols])
im = ax_mean.imshow(mean_attn, cmap='viridis', aspect='auto', interpolation='nearest')
plt.colorbar(im, ax=ax_mean, fraction=0.02)
ax_mean.set_title(
    f'Mean Attention (across {n_heads_actual} heads) — Block {n_blocks}\n'
    f'Sample: "{true_label}"',
    fontweight='bold', fontsize=11
)
ax_mean.set_xlabel('Key timestep (source)')
ax_mean.set_ylabel('Query timestep (target)')

# Annotation: most-attended timestep for the mean
peak_q, peak_k = np.unravel_index(mean_attn.argmax(), mean_attn.shape)
ax_mean.scatter(peak_k, peak_q, color='red', s=60, zorder=5, label='Peak attention')
ax_mean.legend(fontsize=8, loc='upper right')

# Per-head attention maps
for h in range(n_heads_actual):
    row = 1 + h // ncols
    col = h %  ncols
    ax  = fig.add_subplot(gs[row, col])
    ax.imshow(last_attn[h], cmap='plasma', aspect='auto', interpolation='nearest')
    ax.set_title(f'Head {h+1}', fontsize=9)
    ax.set_xlabel('Key t', fontsize=7)
    ax.set_ylabel('Query t', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(
    f'Self-Attention Weights — Last Transformer Block\n'
    f'Gesture: "{true_label}"  |  Sequence length: {N_TIMESTEPS} steps',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_attention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 2: Temporal attention profile ─────────────────────────────────────
# For each timestep t, how much total attention does t RECEIVE (column sum)?
# High values → timestep is important context that other steps attend to
attn_received = mean_attn.sum(axis=0)   # (T,)  sum of attention into each timestep
attn_sent     = mean_attn.sum(axis=1)   # (T,)  sum of attention from each timestep

time_axis_ms  = np.arange(N_TIMESTEPS) / FS_HZ * 1000  # ms

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

ax1.fill_between(time_axis_ms, attn_received, color=TEAL, alpha=0.7, label='Attention received')
ax1.set_ylabel('Attention weight (sum)')
ax1.set_title('Attention Received per Timestep — how important each moment is as context',
              fontsize=10, fontweight='bold')
ax1.axvline(time_axis_ms[attn_received.argmax()], color=RUST, lw=1.5, linestyle='--')
ax1.text(time_axis_ms[attn_received.argmax()], attn_received.max() * 0.9,
         f'{time_axis_ms[attn_received.argmax()]:.0f} ms', fontsize=8, color=RUST)
ax1.legend(fontsize=9)
ax1.grid(True)

ax2.fill_between(time_axis_ms, attn_sent, color=RUST, alpha=0.7, label='Attention sent')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Attention weight (sum)')
ax2.set_title('Attention Sent per Timestep — how broadly each moment looks for context',
              fontsize=10, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True)

plt.suptitle(f'Temporal Attention Profile — "{true_label}"', fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig(RESULTS_DIR / 'transformer_temporal_attention.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Figure 3: All-blocks mean attention evolution ─────────────────────────────
if n_blocks > 1:
    fig, axes = plt.subplots(1, n_blocks, figsize=(5 * n_blocks, 4.5))
    if n_blocks == 1:
        axes = [axes]
    for bi, (attn_blk, ax) in enumerate(zip(attn_weights_list, axes)):
        m = attn_blk.mean(axis=0)
        im = ax.imshow(m, cmap='viridis', aspect='auto', interpolation='nearest')
        ax.set_title(f'Block {bi+1} mean', fontweight='bold')
        ax.set_xlabel('Key t')
        ax.set_ylabel('Query t')
        plt.colorbar(im, ax=ax, fraction=0.04)

    plt.suptitle(
        f'Mean Attention per Encoder Block — "{true_label}"',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'transformer_block_attention_evolution.png',
                dpi=150, bbox_inches='tight')
    plt.show()

print("Attention visualisations saved to results/.")

In [ ]:
# Cell 11 — Save model and results JSON

import datetime

# ── Final metrics ─────────────────────────────────────────────────────────────
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss,   val_acc   = model.evaluate(X_val,   y_val,   verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

print(f"Train : loss={train_loss:.4f}, acc={train_acc:.4f}")
print(f"Val   : loss={val_loss:.4f},   acc={val_acc:.4f}")
print(f"Test  : loss={test_loss:.4f},  acc={test_acc:.4f}")

# ── Full model save ───────────────────────────────────────────────────────────
final_model_path = SAVED_MODELS_DIR / 'transformer_final.keras'
model.save(str(final_model_path))
print(f"\nFull model saved to: {final_model_path}")

# ── SavedModel format (TFLite / TF Serving compatible) ────────────────────────
saved_model_path = SAVED_MODELS_DIR / 'transformer_savedmodel'
model.export(str(saved_model_path))
print(f"SavedModel exported to: {saved_model_path}")

# ── Results JSON ─────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report
report_dict = classification_report(
    y_test, y_pred, target_names=labels, output_dict=True
)

results = {
    "model": "STFTnet_Transformer",
    "timestamp": datetime.datetime.now().isoformat(),
    "config": {
        "data_root": DATA_ROOT,
        "filter_type": FILTER_TYPE,
        "target_len": TARGET_LEN,
        "normalization": NORMALIZATION,
        "per_sample_norm": PER_SAMPLE_NORM,
        "use_mfa_block": USE_MFA_BLOCK,
        "mfa_gamma": MFA_GAMMA,
        "d_model": D_MODEL,
        "n_layers": N_LAYERS,
        "n_heads_requested": N_HEADS,
        "n_heads_actual": N_HEADS_ACTUAL,
        "ffn_dim": FFN_DIM,
        "transformer_dropout": TRANSFORMER_DROPOUT,
        "dense_units": DENSE_UNITS,
        "dropout_rate": DROPOUT_RATE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "lr_warmup_steps": LR_WARMUP_STEPS,
        "use_positional_encoding": USE_POSITIONAL_ENCODING,
        "optimizer": "AdamW" if TFA_AVAILABLE else "Adam",
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "random_seed": RANDOM_SEED,
    },
    "dataset": {
        "n_samples": int(X.shape[0]),
        "n_timesteps": int(X.shape[1]),
        "n_features": int(X.shape[2]),
        "n_classes": int(N_CLASSES),
        "classes": labels,
        "n_train": int(len(X_train)),
        "n_val": int(len(X_val)),
        "n_test": int(len(X_test)),
    },
    "metrics": {
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
        "best_val_accuracy": float(best_val_acc),
        "best_val_epoch": int(best_ep),
        "epochs_trained": int(len(history.history['loss'])),
    },
    "classification_report": report_dict,
    "training_history": {
        "loss":     [float(v) for v in history.history['loss']],
        "accuracy": [float(v) for v in history.history['accuracy']],
        "val_loss":     [float(v) for v in history.history['val_loss']],
        "val_accuracy": [float(v) for v in history.history['val_accuracy']],
    },
    "artifacts": {
        "model_keras":     str(final_model_path),
        "model_savedmodel": str(saved_model_path),
        "training_curves": str(RESULTS_DIR / 'transformer_training_curves.png'),
        "confusion_matrix": str(RESULTS_DIR / 'transformer_confusion_matrix.png'),
        "attention_heatmap": str(RESULTS_DIR / 'transformer_attention_heatmap.png'),
        "temporal_attention": str(RESULTS_DIR / 'transformer_temporal_attention.png'),
        "training_log":    str(RESULTS_DIR / 'transformer_training_log.csv'),
    }
}

results_path = RESULTS_DIR / 'transformer_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults JSON saved to: {results_path}")

# ── Final summary table ───────────────────────────────────────────────────────
print("\n" + "═" * 55)
print("  FINAL RESULTS SUMMARY")
print("═" * 55)
print(f"  Model         : STFTnet-inspired Transformer")
print(f"  D_MODEL       : {D_MODEL}  |  N_LAYERS: {N_LAYERS}  |  N_HEADS: {N_HEADS_ACTUAL}")
print(f"  MFA Block     : {'Yes' if USE_MFA_BLOCK else 'No'}")
print(f"  Pos Encoding  : {'Yes' if USE_POSITIONAL_ENCODING else 'No'}")
print(f"  Features      : {N_FEATURES}  |  Timesteps: {N_TIMESTEPS}")
print(f"  Classes       : {N_CLASSES}")
print(f"  ─────────────────────────────────────────────────")
print(f"  Train Accuracy: {train_acc*100:.2f}%")
print(f"  Val   Accuracy: {val_acc*100:.2f}%")
print(f"  Test  Accuracy: {test_acc*100:.2f}%")
print("═" * 55)